# DID_disease

> 对应代码：`EconometricsCode/code_in_notes/Chap.15/DID_disease.do`（本 notebook 中的 Stata 代码与 do 文件一致）
>
> 运行方式：通过 *PyStata* 在 Jupyter 中调用 Stata，安装与使用说明见 `notebooks/PyStata.md`。

**说明**：为在 notebook 中直接内联显示图形，本 notebook 的 Stata 代码省略了 `.do` 文件中的 `graph export` 导出命令（`code_in_notes` 中对应的 .do 文件保留导出，用于生成书中的插图）。


首先启动 Stata 环境，并把工作目录切换到 `code_in_notes/Chap.15`，这样 Stata 中的相对路径（如 `../datasets/...`）与同名 do 文件完全一致，图片也会输出到该目录。

In [ ]:
# 配置 PyStata：启动 Stata 并注册 %%stata 魔法命令
import os
os.chdir("../../code_in_notes/Chap.15")

import stata_setup
stata_setup.config("/opt/Stata", "mp")


读入流脑疫情与教育性别差距数据。注释说明了变量含义：year 为调查轮数、district 为地区、edu_years 为教育年限、year_birth 为出生年份（1960–1992），并注明流行性脑膜炎发生于 1986 年。

In [ ]:
%%stata
clear
set more off
use ../datasets/disease_gender_gap
*****
** 变量说明
* year：调查的轮数
* district：区
* edu_years：教育年限
* year_birth：出生年龄，1960-1992  **注，流行性脑膜炎发生在1986


按 1986 年疫情发生时的年龄划分三个出生队列（cohort）：1981–1986 年出生（疫情时 0–5 岁，尚未上学）、1974–1980 年（6–12 岁，正处于小学阶段）、1966–1973 年（13–20 岁，中学阶段）。直觉上，疫情冲击应主要影响当时正处于学龄期的队列。

In [ ]:
%%stata
** 作者选取的cohort: 在1986年时0-5岁（不上学），6-12岁（小学），13-20岁（中学）
** 相应的年龄为1981-1986、1974-1980、1966-1973
gen cohort1=(year_birth>=1981 & year_birth<=1986)
gen cohort2=(year_birth>=1974 & year_birth<=1980)
gen cohort3=(year_birth>=1966 & year_birth<=1973)


构造实质的 DID 变量：各队列虚拟变量与地区流脑严重程度 menin_avg 的交叉项，刻画“疫情严重地区 × 受影响队列”的处理强度；再乘 female 得到三重交叉项，用于检验疫情对女性教育相对男性的差异化影响。

In [ ]:
%%stata
** 交叉项
gen co1menin=cohort1*menin_avg
gen co2menin=cohort2*menin_avg
gen co3menin=cohort3*menin_avg

gen co1menin_female=cohort1*female*menin_avg
gen co2menin_female=cohort2*female*menin_avg
gen co3menin_female=cohort3*female*menin_avg


district 是字符串，不能直接用 i. 前缀生成虚拟变量；我们利用行号 _n 与 egen 的 by() 选项，把每个地区映射为其首个观测的行号，得到数值型 district_id。

In [ ]:
%%stata
** district 为字符串，需要变成数字才能用i.district
gen n=_n
egen district_id=min(n), by(district)


第一个回归是实质的 DID：这里的“时间”维度其实是出生队列，“组别”变量是地区疫情严重程度 menin_avg，三个队列都是处理组，其余出生年份充当对照。reghdfe 吸收年份、出生年份与地区三层固定效应，标准误聚类到地区层面。注意局部宏 `control' 从未定义，展开为空，是作者有意预留的控制变量位置。

In [ ]:
%%stata
** 第一个回归，才是实质上的DID，传统的时间其实是cohort，group变量是menin，三个cohort都是处理组，有两个未处理组
reghdfe edu_years co1menin co2menin co3menin female `control', absorb(i.year i.year_birth i.district_id) cl(district_id)


第二个回归加入队列×疫情×女性的三重交叉项，比较不同性别的处理效应：co*menin_female 的系数刻画疫情对女性相对男性的额外教育损失，即 DID 的性别异质性。

In [ ]:
%%stata
** 第二个回归，其实是比较了不同性别的treatment effects
reghdfe edu_years co1menin_female co2menin_female co3menin_female co1menin co2menin co3menin female `control', absorb(i.year i.year_birth i.district_id)  cl(district_id)


留给读者的思考：为什么不控制个体固定效应 i.indvidual_id？（个体固定效应会吸收掉队列层面的大部分变异，与 cohort×menin 交叉项共线。）

In [ ]:
%%stata
******问题：
** 为什么不控制个体的固定效应？i.indvidual_id？


稳健性检查：preserve/restore 块内重复上述两个回归，并 drop 1986 年以后出生的样本——作者意在检验剔除疫情后出生的队列后结论是否依然稳健；restore 之后数据恢复原样。

In [ ]:
%%stata
** 如果不看1986以后出生的？依然是稳健的
preserve
reghdfe edu_years co1menin co2menin co3menin female `control', absorb(i.year i.year_birth i.district_id) cl(district_id)
reghdfe edu_years co1menin_female co2menin_female co3menin_female co1menin co2menin co3menin female `control', absorb(i.year i.year_birth i.district_id)  cl(district_id)
drop if year_birth>1986
restore


若担心同一年龄段内的截面相关（例如同龄人的 peer effects），把聚类改为 district_id 与 year_birth 的双向聚类，允许误差项在地区与出生年份两个维度上相关。

In [ ]:
%%stata
** 如果考虑截面相关，比如同一年龄段的peer effects，该cluster什么？
reghdfe edu_years co1menin co2menin co3menin female `control', absorb(i.year i.year_birth i.district_id) cl(district_id year_birth)
reghdfe edu_years co1menin_female co2menin_female co3menin_female co1menin co2menin co3menin female `control', absorb(i.year i.year_birth i.district_id)  cl(district_id year_birth)


更严格的固定效应设定：把 absorb 改为 i.year 与 i.year_birth#i.district_id，即“地区×出生年份”交互固定效应，允许每个地区有自己任意的队列变化模式，识别仅来自同一地区内部不同队列之间的比较。

In [ ]:
%%stata
** 如果继续严格控制固定效应
reghdfe edu_years co1menin co2menin co3menin female `control', absorb(i.year i.year_birth#i.district_id) cl(district_id year_birth)
reghdfe edu_years co1menin_female co2menin_female co3menin_female co1menin co2menin co3menin female `control', absorb(i.year i.year_birth#i.district_id)  cl(district_id year_birth)


两个思考问题：在地区×出生年份固定效应下，第一个回归的系数还能识别出来吗？第二个回归中三重交叉项识别的又是什么变异？

In [ ]:
%%stata
* 问题：第一个回归方程能估计出来吗？
* 问题：第二个回归方程识别出来的是什么？
